# Part 5: The Analyst Report

After you have successfully deployed your pipeline and run the **Burst** profile (500 messages) in the test apparatus, you need to extract the results and answer a few questions.

We use `boto3` to scan the DynamoDB table, handling pagination automatically, and convert the results into standard Python dictionaries and floats.

## Setup: Configure Your Student ID
Replace `YOURID` below with the exact student ID you used for deployment.

In [5]:
%pip install boto3
STUDENT_ID = "sthomas"  # <--- Change this
TABLE_NAME = f"adflow-{STUDENT_ID}-results"
REGION = "us-east-1"
print(f"Target Table: {TABLE_NAME}")

Target Table: adflow-sthomas-results


In [18]:
import os
import boto3

# Set AWS credentials
os.environ["AWS_ACCESS_KEY_ID"] = "your access key"
os.environ["AWS_SECRET_ACCESS_KEY"] = "your secret access key"
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

# ONLY include this if your professor gave you a session token
# os.environ["AWS_SESSION_TOKEN"] = "YOUR_SESSION_TOKEN"

# Create session
session = boto3.Session(
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name=os.environ["AWS_DEFAULT_REGION"]
)

# Test credentials
sts = session.client("sts")
print(sts.get_caller_identity())

{'UserId': 'AIDASGFHDAHNRNAOEKSOJ', 'Account': '150674276827', 'Arn': 'arn:aws:iam::150674276827:user/adflow-tester', 'ResponseMetadata': {'RequestId': 'b4e4dcf5-4dd7-4bd1-a6a2-a9a574514b1e', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'b4e4dcf5-4dd7-4bd1-a6a2-a9a574514b1e', 'x-amz-sts-extended-request-id': 'MTp1cy1lYXN0LTE6UzoxNzc5MTM4NDAwODgyOlI6VTBNUkZSVVc=', 'content-type': 'text/xml', 'content-length': '410', 'date': 'Mon, 18 May 2026 21:06:40 GMT'}, 'RetryAttempts': 0}}


## Step 1: Export Data from DynamoDB
This cell connects to your DynamoDB table, downloads all records, and converts the Decimal values back to standard floats.

In [19]:
import boto3
from decimal import Decimal
from collections import Counter

# Note: This uses your active AWS credentials (from `aws configure` or exported environment variables)
dynamodb = boto3.resource("dynamodb", region_name=REGION)
table = dynamodb.Table(TABLE_NAME)

results = []
response = table.scan()
results.extend(response.get("Items", []))

# Handle pagination if the table has more than 1 MB of data
while "LastEvaluatedKey" in response:
    response = table.scan(ExclusiveStartKey=response["LastEvaluatedKey"])
    results.extend(response.get("Items", []))

print(f"\nLoaded {len(results)} records from DynamoDB.")

# Convert Decimal types to Python floats for easier math/plotting
for item in results:
    for key in ["winning_bid_amount", "winning_score", "score_margin"]:
        if key in item and isinstance(item[key], Decimal):
            item[key] = float(item[key])

if results:
    print("\nSample record:")
    print(results[0])


Loaded 500 records from DynamoDB.

Sample record:
{'processed_at': '2026-05-18T20:50:12.536517+00:00', 'opportunity_id': 'bff00546-23d3-44ef-9616-207f4c52a9a6', 'winning_score': 8.69375, 'score_margin': 1.7437500000000004, 'winning_advertiser_id': 'adv_energy_01', 'winning_bid_amount': 5.35, 'content_category': 'sports'}


## Section 1: Pipeline Evidence
Print the total records and a quick count of auction wins per advertiser across the entire dataset to prove your pipeline successfully routed messages.

In [21]:
# Total number of records
print(f"Total pipeline records: {len(results)}")

# Auction wins per advertiser (overall)
winner_counts = Counter(r["winning_advertiser_id"] for r in results)
print("\nWins per advertiser (overall):")
for advertiser, count in winner_counts.most_common():
    print(f"  {advertiser}: {count}")

Total pipeline records: 500

Wins per advertiser (overall):
  adv_auto_01: 67
  adv_fintech_01: 59
  adv_insurance_01: 56
  adv_travel_01: 45
  adv_streaming_01: 44
  adv_fastfood_01: 34
  adv_energy_01: 29
  adv_auto_02: 26
  adv_sportswear_01: 25
  adv_fastfood_02: 21
  adv_insurance_02: 18
  adv_beauty_01: 14
  adv_gaming_01: 11
  adv_fintech_02: 11
  adv_telecom_01: 10
  adv_travel_02: 10
  adv_energy_02: 6
  adv_streaming_02: 6
  adv_sportswear_02: 3
  adv_retail_01: 3
  adv_gaming_02: 1
  adv_beauty_02: 1


**Evidence Requirement:** Don't forget to push a screenshot of the **Test Apparatus** (showing a completed Burst run) to a `screenshots/` directory in this repo when submitting.

---
## Q1: Results Analysis

**Question:** Which advertiser won the most auctions overall? Which advertiser won the most in the `sports` content category specifically? Why do the overall and sports-specific rankings differ? Explain in 2–3 sentences, referencing the relevance multiplier table.

In [22]:
# Sports category records
sports_results = [r for r in results if r.get("content_category") == "sports"]
print(f"Sports records: {len(sports_results)}")

# Overall top winner
top_overall = winner_counts.most_common(1)[0]
print(f"\nOverall top winner: {top_overall[0]} with {top_overall[1]} wins")

# Sports top winner
sports_counts = Counter(r["winning_advertiser_id"] for r in sports_results)
top_sports = sports_counts.most_common(1)[0]
print(f"Sports top winner: {top_sports[0]} with {top_sports[1]} wins")

Sports records: 135

Overall top winner: adv_auto_01 with 67 wins
Sports top winner: adv_auto_01 with 24 wins


**Your Answer (Q1):**

The overall top winner was adv_auto_01 with 67 wins, and it was also
the top winner in the sports category specifically with 24 wins out of
135 sports records. The overall and sports-specific rankings are the
same in this case because adv_auto_01 consistently submits competitive
bids across all content categories. However, the relevance multiplier
still shapes the broader rankings — for example, adv_fintech_01 came
second overall with 59 wins largely because finance content applies a
1.5× multiplier to fintech advertisers, the highest multiplier in the
table. Similarly, adv_insurance_01 placed third with 56 wins due to the
1.3× boost it receives on finance content. By contrast, adv_sportswear_01
only managed 25 wins overall despite having the strongest multiplier on
sports content (1.4×), because sports was just one of five content
categories — without a matching bonus in other categories, its raw bids
were not competitive enough to win broadly.

---
## Q2: Code Reflection

Answer **one** of the following (your choice):

* **Option A (Scale & Limits):** The test apparatus sent messages in small batches. If traffic suddenly spiked from 10 opportunities a second to 10,000 a second, what specific components of our current pipeline (SQS limits, Lambda concurrency, DynamoDB throughput) would become bottlenecks first, and what AWS settings would you adjust to handle the load?
* **Option B (The Distributed Process):** Writing code for an event-driven, queue-based pipeline is very different from writing a single local script. What was the most challenging part of getting SQS, Lambda, and DynamoDB to communicate correctly, or the most confusing bug you encountered, and what did it teach you about distributed architecture?

A well-argued two-paragraph response is sufficient for either option.

**Your Answer (Q2):**

The most challenging part of this pipeline was understanding that each
AWS service has its own data type contract, and data must be explicitly
transformed at every boundary. The clearest example was the float vs
Decimal problem between SQS and DynamoDB — SQS requires a JSON-serialized
string, which works fine with Python floats, but DynamoDB rejects floats
entirely and requires every numeric value to be wrapped in Decimal(str(value)).
This meant maintaining two separate versions of the same result record,
one for each service, which is not something you would ever encounter
writing a single local script.

This experience taught me that distributed architecture is less about
writing complex logic and more about carefully managing the interfaces
between components. A bug at any one boundary — a missing field, a wrong
type, a misspelled key like batchItemFailures — does not throw an obvious
error. SQS will silently retry an entire batch if the failure response
format is wrong, and DynamoDB will reject a write without explaining why
the type is invalid unless you know to look for it. Debugging distributed
systems requires thinking in terms of contracts between services rather
than just the logic inside a single function.